# 第 4 周练习：测试生成（Python）

用 LLM 为代码生成测试，并配合 Gradio 等界面进行交互演示。

## 练习流水线

1. 选模型 → 给函数补 docstring / 注释（不改逻辑）
2. 再生成恰好 5 条 JSON 测试用例（含边界）
3. 在本地 `exec` 跑用例，把 PASS/FAIL 打到界面

## 怎么跑

1. `.env` 准备好 `OPENAI_API_KEY` / `GOOGLE_API_KEY` / `GROQ_API_KEY`（按你要用的后端）
2. 若用 Ollama 云端模型名，确保本机 Ollama 可用
3. 依次运行单元格后 `demo.launch`，粘贴函数并填写函数名


In [ ]:
# ========== 导入依赖：环境、OpenAI 兼容客户端、Gradio、JSON/异常工具 ==========

# 导入标准库 os：读各家 API Key 环境变量
import os
# 导入标准库 io：本练习可能用于缓冲输出（保持原导入）
import io
# 导入标准库 sys：进程/路径相关工具（保持原导入）
import sys
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进环境
from dotenv import load_dotenv
# 从 openai 导入 OpenAI：统一用 OpenAI 兼容协议连多家后端
from openai import OpenAI
# 导入 gradio：搭「代码 → 文档 → 用例 → 结果」流水线界面
import gradio as gr
# 导入 subprocess：子进程工具（保持原导入，即使本格未直接调用）
import subprocess
# 从 IPython.display 导入展示工具（笔记本里可漂亮显示 Markdown）
from IPython.display import Markdown, display
# 导入标准库 json：解析模型返回的测试用例 JSON
import json
# 导入 traceback：exec/测试失败时打印完整堆栈
import traceback


In [ ]:
# ========== 加载环境变量：三家密钥分别读出 ==========

# 加载 .env；override=True 覆盖已存在的同名环境变量
load_dotenv(override=True)
# OpenAI 官方密钥
openai_api_key = os.getenv('OPENAI_API_KEY')
# Google Gemini（走 OpenAI 兼容端点时用）
google_api_key = os.getenv('GOOGLE_API_KEY')
# Groq 平台密钥
groq_api_key = os.getenv('GROQ_API_KEY')


In [ ]:
# ========== 多后端客户端：URL + OpenAI() + 模型名到客户端的映射 ==========

# Gemini 的 OpenAI 兼容基址（URL 勿改）
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
# 本机 Ollama OpenAI 兼容基址
ollama_url = "http://localhost:11434/v1"
# Groq OpenAI 兼容基址
groq_url = "https://api.groq.com/openai/v1"


# ---- 客户端实例：默认读环境里的 OPENAI_API_KEY；其它显式传 key/base_url ----
openai_client = OpenAI()
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
# Ollama 常需任意非空 api_key 字符串；真实鉴权在本地
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)

# 下拉可选的模型 id 列表（字符串必须与后端登记名一致）
models = [
    "gpt-5-nano",
    "gemini-2.5-flash", # Change this
    "openai/gpt-oss-120b",
    "llama-3.3-70b-versatile",
    "gemma4:31b-cloud",
    "qwen3-coder-next:cloud" # change this
]

# 模型名 → 该用哪个客户端实例
clients = {
    "gpt-5-nano": openai_client,
    "gemini-2.5-flash": gemini,
    "openai/gpt-oss-120b": groq,
    "llama-3.3-70b-versatile": groq,
    "gemma4:31b-cloud": ollama,
    "qwen3-coder-next:cloud": ollama
}


In [ ]:
# ========== 两段 system prompt：补文档 vs 生成 JSON 测试（正文勿改）==========

# 角色：资深 Python；只加 docstring/注释，禁止改逻辑与重命名
DOCS_SYSTEM_PROMPT = """
You are a senior Python developer.

Your task:
- Add clear docstrings to all functions and classes
- Add meaningful inline comments
- Do NOT change the logic of the code
- Do NOT rename variables or functions
- Keep code executable

Return ONLY the updated Python code.
"""


# 角色：测试专家；恰好 5 条用例，只返回合法 JSON 数组
TESTCASE_SYSTEM_PROMPT = """
You are a strict Python testing expert.

Given a Python function:
1. Generate EXACTLY 5 test cases
2. Focus on edge cases and tricky inputs
3. Each list must contain at most 10 elements
4. DO NOT generate very large lists
5. Include:
   - Normal case
   - Edge case
   - Corner case
   - Invalid input (if applicable)
   - Stress/difficult case
6. Use ONLY standard ASCII characters
7. Do NOT use fancy quotes or special unicode
8. Do NOT wrap in markdown (no ```)

Return ONLY valid JSON. No markdown. No explanation.

Format:
[
  {
    "input": [args],
    "expected_output": value,
    "description": "..."
  }
]

IMPORTANT:
- Ensure expected_output is CORRECT
- No explanations outside JSON
"""


In [ ]:
# ========== 步骤 1：调用模型给代码补 docstring / 行内注释 ==========

# code：用户源码；model_name：同时用作 clients 键和 API 的 model 参数
def add_docs_and_comments(code, model_name):
    # 按模型名取出对应 OpenAI 兼容客户端
    client = clients[model_name]

    # Chat Completions：system 定规则，user 放原始代码
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": DOCS_SYSTEM_PROMPT},
            {"role": "user", "content": code}
        ]
    )

    # 返回助手生成的「带文档代码」文本
    return response.choices[0].message.content


In [ ]:
# ========== 清洗：去掉 markdown 围栏并规范化缩进，便于 exec ==========

# 正则：匹配/替换代码围栏
import re
# textwrap.dedent：去掉公共前导空白，避免缩进导致 SyntaxError
import textwrap

# 把模型可能包了 ``` 的输出变成可 exec 的纯 Python
def clean_code(code: str) -> str:
    # 去掉 ```python 与结尾 ```
    code = re.sub(r"```python|```", "", code)

    # 去掉整块公共缩进
    code = textwrap.dedent(code)

    # 去首尾空白后返回
    return code.strip()


In [ ]:
# ========== 步骤 2：让模型产出 JSON 测试用例并解析 ==========

# 输入（通常是已补文档的代码）+ 模型名 → Python list[dict]
def generate_test_cases(code, model_name):
    # 取对应客户端
    client = clients[model_name]

    # 用 TESTCASE_SYSTEM_PROMPT 约束「只返回 JSON」
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": TESTCASE_SYSTEM_PROMPT},
            {"role": "user", "content": code}
        ]
    )

    # 取出模型文本
    content = response.choices[0].message.content

    try:
        # 解析为 Python 对象（期望是 list）
        return json.loads(content)
    except:
        # 解析失败：抛出带原文的错误（错误文案保持英文）
        raise ValueError("Invalid JSON from model:\n" + content)


In [ ]:
# ========== 步骤 3：exec 源码并逐条跑测试用例 ==========

# code：待测源码；test_cases：JSON 解析后的列表；function_name：入口函数名
def run_tests(code, test_cases, function_name):
    # 隔离的命名空间：exec 定义的函数会进这里
    local_env = {}
    # 累积给人看的测试日志
    output_log = ""

    # 先清洗围栏/缩进，避免模型包了 markdown
    code = clean_code(code)

    try:
        # 在 local_env 里执行定义语句
        exec(code, local_env)
    except Exception:
        # 代码本身跑不起来：返回堆栈
        return "❌ Code execution failed:\n" + traceback.format_exc()

    # 按用户填写的函数名取出可调用对象
    func = local_env.get(function_name)

    if not func:
        # 名字对不上：直接失败提示
        return f"❌ Function '{function_name}' not found."

    # 从 1 开始编号，遍历每条用例
    for i, test in enumerate(test_cases, 1):
        # 位置参数列表
        inputs = test["input"]
        # 期望输出（或期望异常名字符串）
        expected = test["expected_output"]
        # 可选描述
        desc = test.get("description", "")

        try:
            # 解包调用：func(*inputs)
            result = func(*inputs)

            # 若期望是带 "Error" 的字符串，说明本该抛异常却返回了值 → FAIL
            if isinstance(expected, str) and "Error" in expected:
                passed = False
                output_log += f"\nTest Case {i}: {desc}\n"
                output_log += f"Input: {inputs}\n"
                output_log += f"Expected Exception: {expected}\n"
                output_log += f"Returned: {result}\n"
                output_log += "❌ FAIL (Expected exception but got value)\n"
                continue

            # 普通相等比较
            passed = result == expected

            output_log += f"\nTest Case {i}: {desc}\n"
            output_log += f"Input: {inputs}\n"
            output_log += f"Expected: {expected}\n"
            output_log += f"Returned: {result}\n"
            output_log += "✅ PASS\n" if passed else "❌ FAIL\n"

        except Exception as e:
            # 抛异常时：若期望字符串出现在异常类名里，算 PASS
            if isinstance(expected, str) and expected in type(e).__name__:
                output_log += f"\nTest Case {i}: {desc}\n"
                output_log += f"Input: {inputs}\n"
                output_log += f"Expected Exception: {expected}\n"
                output_log += f"Got Exception: {type(e).__name__}\n"
                output_log += "✅ PASS\n"
            else:
                # 非预期异常：记 ERROR + 堆栈
                output_log += f"\nTest Case {i}: {desc}\n"
                output_log += f"Input: {inputs}\n"
                output_log += "❌ ERROR\n"
                output_log += traceback.format_exc()

    # 返回完整日志字符串给界面
    return output_log


In [ ]:
# ========== 流水线生成器：分步 yield，让 Gradio 渐进刷新三块输出 ==========

# Gradio 支持 generator：每次 yield (文档代码, 用例 JSON 文本, 状态/结果)
def process_code(code, model_name, function_name):
    try:
        # Step 1：补 docstring / 注释
        documented_code = add_docs_and_comments(code, model_name)

        # 立刻刷新 UI：已有文档代码，用例与结果尚空/占位
        yield documented_code, "", "⏳ Generating test cases..."

        # Step 2：基于「带文档代码」生成用例（更贴近最终要测的文本）
        test_cases = generate_test_cases(documented_code, model_name)
        # 美化为缩进 JSON 字符串，便于右侧 Code 展示
        test_cases_str = json.dumps(test_cases, indent=2)

        # 再次刷新：用例已就绪，准备跑测
        yield documented_code, test_cases_str, "⏳ Running tests..."

        # Step 3：先对 documented_code 跑测；失败则回退用原始 code
        try:
            results = run_tests(documented_code, test_cases, function_name)
        except:
            results = run_tests(code, test_cases, function_name)

        # 最终三元组：文档代码 + 用例 + 结果日志
        yield documented_code, test_cases_str, results

    except Exception as e:
        # 任一步炸掉：三块都显示错误信息
        yield "ERROR", "ERROR", str(e)


In [ ]:
# ========== Gradio Blocks：左输入右输出，一键跑完整流水线 ==========

with gr.Blocks() as demo:
    # 页标题
    gr.Markdown("# 🧪 LLM Code Tester")

    with gr.Row():
        # 左栏：用户输入
        with gr.Column(scale=1):
            # 粘贴待测 Python 函数
            code_input = gr.Textbox(
                label="Python Code",
                lines=20,
                placeholder="Paste your Python function here..."
            )

            # 告诉 run_tests 要从命名空间取哪个函数名
            function_name = gr.Textbox(
                label="Function Name",
                placeholder="e.g. add"
            )

            # 模型下拉；默认第一项
            model_dropdown = gr.Dropdown(
                choices=models,
                label="Select Model",
                value=models[0]
            )

            # 启动 generator 流水线
            run_button = gr.Button("Run Pipeline")

        # 右栏：三块输出
        with gr.Column(scale=1):
            # 补完文档的代码
            doc_output = gr.Code(label="📘 Documented Code")

            # JSON 用例
            test_output = gr.Code(label="🧾 Test Cases (JSON)")

            # PASS/FAIL 日志
            result_output = gr.Textbox(
                label="📊 Test Results",
                lines=20
            )

    # 绑定点击：process_code 的三次 yield 会渐进更新三块
    run_button.click(
        fn=process_code,
        inputs=[code_input, model_dropdown, function_name],
        outputs=[doc_output, test_output, result_output]
    )

# 启动本地 Gradio 服务
demo.launch()
